# Adaptive Testing with Scorebook and Local Inference

This notebook demonstrates how to use Scorebook's adaptive evaluation capabilities with local transformer models instead of cloud APIs.

## What is Adaptive Testing?

Adaptive testing dynamically adjusts question difficulty based on model performance, providing more efficient and accurate capability assessment.

## Prerequisites

- Trismik API key
- Trismik project ID
- Google Colab with GPU (recommended) or CPU runtime

## 1. Install Dependencies

In [24]:
# Install required packages
!pip install scorebook trismik transformers torch accelerate nest_asyncio -q

# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


GPU Available: False


## 2. Import Libraries

In [25]:
import os
import re
import string
from typing import Any, Dict, List
from getpass import getpass
import json
import secrets

import nest_asyncio
import transformers
import trismik
from scorebook import evaluate, login
from scorebook.inference_pipeline import InferencePipeline


nest_asyncio.apply()

# Set transformers verbosity
transformers.utils.logging.set_verbosity_error()

## 3. Authentication Setup

Enter your Trismik credentials to enable adaptive evaluation.

In [3]:
# Get Trismik credentials
TRISMIK_API_KEY = getpass("Enter your Trismik API Key: ")

Enter your Trismik API Key:  ········


In [16]:
client = trismik.client_async.TrismikAsyncClient(api_key=TRISMIK_API_KEY)
project = await client.create_project(f"Demo_Project_{secrets.token_hex(4)}")
experiment_name = f"Demo_Experiment_{secrets.token_hex(4)}"

In [17]:
# Login to Trismik
login(TRISMIK_API_KEY)
print("✓ Successfully logged in to Trismik")

✓ Successfully logged in to Trismik


## 4. Load Local Model

We'll use Microsoft's Phi-4-mini-instruct model for this example. You can replace this with any HuggingFace model.

In [18]:
# Model configuration
MODEL_NAME = "microsoft/Phi-4-mini-instruct"

print(f"Loading model: {MODEL_NAME}")
print("This may take a few minutes on first run...")

# Initialize the pipeline
pipeline = transformers.pipeline(
    "text-generation",
    model=MODEL_NAME,
    model_kwargs={"torch_dtype": "auto"},
    device_map="auto"
)

print(f"✓ Model loaded successfully")

# Generation parameters for consistent outputs
GENERATION_ARGS = {
    "max_new_tokens": 10,
    "temperature": 0.0,
    "do_sample": False,
    "return_full_text": False,
    "pad_token_id": pipeline.tokenizer.eos_token_id
}

Loading model: microsoft/Phi-4-mini-instruct
This may take a few minutes on first run...


Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:05<00:00,  2.51s/it]


✓ Model loaded successfully


## 5. Define Inference Pipeline Components

The InferencePipeline consists of three components:
1. **Preprocessor**: Formats questions for the model
2. **Inference**: Runs the model
3. **Postprocessor**: Extracts answers from model output

In [19]:
def preprocessor(eval_item: Dict, **hyperparameters: Any) -> List[Dict]:
    """Format evaluation item for HuggingFace model.
    
    Args:
        eval_item: Dictionary containing 'question' and optionally 'options'
        hyperparameters: Additional configuration
    
    Returns:
        Messages formatted for the model
    """
    # Build the prompt
    prompt = eval_item["question"]
    
    # Add multiple choice options if present
    if "options" in eval_item and eval_item["options"]:
        prompt += "\n\nOptions:\n"
        for i, option in enumerate(eval_item["options"]):
            letter = string.ascii_uppercase[i]
            prompt += f"{letter}: {option}\n"
    
    # System instruction for clear answers
    system_message = """
    You are a helpful assistant. For multiple choice questions, answer with ONLY a single letter (A, B, C, D, etc.).
    Do not include any explanation, punctuation, or additional text.
    For other questions, provide a brief, direct answer.
    """.strip()
    
    # Format as messages for the model
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    
    return messages


def inference(preprocessed_items: List[Any], **hyperparameters: Any) -> List[Any]:
    """Run inference on preprocessed items.
    
    Args:
        preprocessed_items: List of formatted messages
        hyperparameters: Additional configuration
    
    Returns:
        List of model outputs
    """
    results = []
    
    for item in preprocessed_items:
        try:
            # Run inference with consistent generation parameters
            output = pipeline(item, **GENERATION_ARGS)
            results.append(output)
        except Exception as e:
            print(f"Error during inference: {e}")
            # Return empty output on error
            results.append([{"generated_text": ""}])
    
    return results


def postprocessor(model_output: Any, **hyperparameters: Any) -> str:
    """Extract answer from model output with retry logic.
    
    Args:
        model_output: Raw model output
        hyperparameters: Additional configuration
    
    Returns:
        Extracted answer as a string
    """
    try:
        # Extract the generated text
        if isinstance(model_output, list) and len(model_output) > 0:
            generated = model_output[0].get("generated_text", "")
        else:
            generated = str(model_output)
        
        # Clean the response
        answer = generated.strip()
        
        # For single letter responses (multiple choice)
        if len(answer) == 1 and answer in string.ascii_uppercase:
            return answer
        
        # Try to extract letter from longer response
        # Pattern: "A" or "A:" or "A." at the beginning
        match = re.match(r'^([A-Z])[:.)]?', answer)
        if match:
            return match.group(1)
        
        # Pattern: "The answer is A" or "Answer: A"
        match = re.search(r'(?:answer|Answer)[:\s]+([A-Z])', answer)
        if match:
            return match.group(1)
        
        # Return the full answer if no pattern matches
        # (for non-multiple choice questions)
        return answer[:100]  # Limit length for safety
        
    except Exception as e:
        print(f"Error in postprocessing: {e}")
        return ""


# Test the pipeline components
print("Testing pipeline components...")
test_item = {
    "question": "What is 2+2?",
    "options": ["3", "4", "5", "6"]
}

test_preprocessed = preprocessor(test_item)
print(f"✓ Preprocessor test passed")

test_output = inference([test_preprocessed])
print(f"✓ Inference test passed")

test_answer = postprocessor(test_output[0])
print(f"✓ Postprocessor test passed")
print(f"Test answer: '{test_answer}'")

Testing pipeline components...
✓ Preprocessor test passed
✓ Inference test passed
✓ Postprocessor test passed
Test answer: 'B'


## 6. Create InferencePipeline and Run Adaptive Evaluation

In [21]:
# Create the InferencePipeline
inference_pipeline = InferencePipeline(
    model=MODEL_NAME,
    preprocessor=preprocessor,
    inference_function=inference,
    postprocessor=postprocessor
)

print("✓ InferencePipeline created")
print(f"Dataset: MMLUPro2025:adaptive")
print(f"Experiment ID: {experiment_name}")
print(f"Project ID: {project.id}")

✓ InferencePipeline created

Starting adaptive evaluation...
Dataset: MMLUPro2025:adaptive
Experiment ID: Demo_Experiment_b8298079
Project ID: 3eadcc11c152c1481111286863bfe71f20a04e74


In [35]:
print("\nStarting adaptive evaluation...")
# Run the adaptive evaluation
results = evaluate(
    inference_pipeline,
    datasets="MMLUPro2024:adaptive",  # Adaptive dataset
    experiment_id=experiment_name,
    project_id=project.id,
    return_dict=True,
    return_aggregates=True,
    return_items=True,
    return_output=True
)

print("\n✓ Evaluation completed!")

Unrecognized dataset type: AdaptiveEvalDataset(name='MMLUPro2024')



Starting adaptive evaluation...


Datasets      0%|                                        | 0/1
Hyperparams   0%|                                        | 0/1
Datasets      0%|                                        | 0/1


✓ Evaluation completed!


## 7. Analyze Results

In [ ]:
# Display aggregate results
if "aggregates" in results:
    print("=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    
    aggregates = results["aggregates"]
    
    # Display overall metrics
    for metric, value in aggregates.items():
        if isinstance(value, (int, float)):
            if metric.endswith("_score") or "accuracy" in metric.lower():
                print(f"{metric}: {value:.2%}")
            else:
                print(f"{metric}: {value}")
        else:
            print(f"{metric}: {value}")
else:
    print("No aggregate results available")

# Show sample of individual results
if "items" in results and len(results["items"]) > 0:
    print("\n" + "=" * 50)
    print("SAMPLE QUESTIONS AND RESPONSES")
    print("=" * 50)
    
    # Display first 3 items
    for i, item in enumerate(results["items"][:3], 1):
        print(f"\nQuestion {i}:")
        print(f"Q: {item.get('question', 'N/A')[:100]}...")
        
        if "output" in item:
            print(f"Model Answer: {item['output']}")
        
        if "label" in item:
            print(f"Correct Answer: {item['label']}")
        
        if "score" in item:
            print(f"Score: {item['score']}")

## 8. Visualize Performance (Optional)

In [ ]:
# Optional: Create a simple visualization of results
try:
    import matplotlib.pyplot as plt
    
    if "items" in results:
        # Calculate accuracy by question position
        scores = [item.get("score", 0) for item in results["items"]]
        
        if scores:
            # Plot accuracy over questions
            plt.figure(figsize=(10, 6))
            plt.plot(range(1, len(scores) + 1), scores, marker='o', alpha=0.6)
            plt.xlabel('Question Number')
            plt.ylabel('Score (1=Correct, 0=Incorrect)')
            plt.title('Adaptive Testing Performance')
            plt.grid(True, alpha=0.3)
            
            # Add rolling average
            if len(scores) > 10:
                window = 10
                rolling_avg = [sum(scores[max(0, i-window):i+1])/len(scores[max(0, i-window):i+1]) 
                               for i in range(len(scores))]
                plt.plot(range(1, len(scores) + 1), rolling_avg, 
                        label=f'{window}-Question Rolling Average', linewidth=2, color='red')
                plt.legend()
            
            plt.tight_layout()
            plt.show()
            
            print(f"\nTotal Questions: {len(scores)}")
            print(f"Overall Accuracy: {sum(scores)/len(scores):.2%}")
            
except ImportError:
    print("Matplotlib not available. Skipping visualization.")

## 9. Save Results (Optional)

In [ ]:
# Save results to file
output_filename = f"adaptive_results_{EXPERIMENT_ID}.json"

# Save to JSON
with open(output_filename, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"✓ Results saved to {output_filename}")

# In Colab, download the file
try:
    from google.colab import files
    files.download(output_filename)
    print(f"✓ File downloaded")
except ImportError:
    print("Not running in Colab - file saved locally")

## Summary

You've successfully run an adaptive evaluation using:
- **Scorebook** for evaluation orchestration
- **Local transformer models** for inference
- **Trismik's adaptive datasets** for intelligent question selection

### Key Takeaways:

1. **Adaptive testing** adjusts difficulty based on model performance
2. **Local inference** provides full control and no API costs
3. **Scorebook's InferencePipeline** makes it easy to swap between local and cloud inference

### Next Steps:

- Try different models by changing `MODEL_NAME`
- Adjust generation parameters for better accuracy
- Compare results with cloud-based inference
- Explore other adaptive datasets available through Trismik